# Druglib.com Preprocessing
Prepare Druglib for ABSA validation and aspect-head supervision.

In [1]:
# DEV MODE keeps notebook runs small during development; set to False for full DGX experiments.
DEV_MODE = True
SAMPLE_SIZE = 10000

# Resolve the project root from either the repository root or notebooks/ directory.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the shared device and seed utilities required by the specification.
from utils.device import RANDOM_SEED, get_device, set_seed

set_seed(RANDOM_SEED)
DEVICE = get_device()


Selected device: CUDA (NVIDIA GeForce RTX 5070 Ti Laptop GPU)


In [2]:
import pandas as pd


# Shared preprocessing utilities used identically across datasets.
import html
import re

LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
NUM_CLASSES = 3


def clean_text(value):
    # Apply the exact text cleaning pipeline from the specification.
    if not isinstance(value, str):
        return ""
    text = value.lower()
    text = html.unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def label_from_rating_10(rating):
    # Map Drugs.com 1-10 ratings to Negative/Neutral/Positive labels.
    rating = float(rating)
    if 1 <= rating <= 4:
        return 0
    if 5 <= rating <= 6:
        return 1
    if 7 <= rating <= 10:
        return 2
    return None


def label_from_rating_5(rating):
    # Map 1-5 ratings to Negative/Neutral/Positive labels.
    rating = float(rating)
    if 1 <= rating <= 2:
        return 0
    if rating == 3:
        return 1
    if 4 <= rating <= 5:
        return 2
    return None


def label_from_effectiveness(value):
    # Map Druglib effectiveness text to the three-class label space.
    mapping = {
        "Ineffective": 0,
        "Marginally Effective": 1,
        "Moderately Effective": 1,
        "Considerably Effective": 2,
        "Highly Effective": 2,
    }
    return mapping.get(value, None)


def label_from_side_effects(value):
    # Map Druglib side-effect severity text to the three-class label space.
    mapping = {
        "Severe Side Effects": 0,
        "Extremely Severe Side Effects": 0,
        "Mild Side Effects": 1,
        "Moderate Side Effects": 1,
        "No Side Effects": 2,
    }
    return mapping.get(value, None)


## Load Raw TSV Files

In [3]:
raw_dir = PROJECT_ROOT / "data/raw"
train_raw = pd.read_csv(raw_dir / "drugLibTrain_raw.tsv", sep="\t")
test_raw = pd.read_csv(raw_dir / "drugLibTest_raw.tsv", sep="\t")
df = pd.concat([train_raw, test_raw], ignore_index=True)
if DEV_MODE:
    df = df.head(SAMPLE_SIZE)
print("Combined shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
print("Null counts:")
print(df.isna().sum())


Combined shape: (4143, 9)
Columns: ['Unnamed: 0', 'urlDrugName', 'rating', 'effectiveness', 'sideEffects', 'condition', 'benefitsReview', 'sideEffectsReview', 'commentsReview']


,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,3947,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,1951,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


Null counts:
Unnamed: 0            0
urlDrugName           0
rating                0
effectiveness         0
sideEffects           0
condition             1
benefitsReview       23
sideEffectsReview    98
commentsReview       13
dtype: int64


## Clean Review Fields and Labels

In [4]:
keep_cols = ["rating", "effectiveness", "sideEffects", "benefitsReview", "sideEffectsReview", "commentsReview"]
df = df[keep_cols].dropna().drop_duplicates().copy()
for column in ["benefitsReview", "sideEffectsReview", "commentsReview"]:
    df[column] = df[column].apply(clean_text)

# Concatenate the three Druglib text fields into one review for model training.
df["review"] = (
    df["benefitsReview"].astype(str) + " " +
    df["sideEffectsReview"].astype(str) + " " +
    df["commentsReview"].astype(str)
).str.strip()
df["label"] = df["rating"].apply(label_from_rating_5)
df["efficacy_label"] = df["effectiveness"].apply(label_from_effectiveness)
df["side_effects_label"] = df["sideEffects"].apply(label_from_side_effects)
df["ease_label"] = -100
df["satisfaction_label"] = -100
df = df.dropna(subset=["label", "efficacy_label", "side_effects_label"]).copy()
for column in ["label", "efficacy_label", "side_effects_label", "ease_label", "satisfaction_label"]:
    df[column] = df[column].astype(int)


## Verify and Save

In [5]:
for column in ["label", "efficacy_label", "side_effects_label"]:
    print(f"\n{column} distribution")
    print(df[column].value_counts(normalize=True).sort_index())

output_path = PROJECT_ROOT / "data/processed/druglib_clean.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", df.shape)
display(df.head())



label distribution
label
0    0.489852
1    0.174354
2    0.335793
Name: proportion, dtype: float64

efficacy_label distribution
efficacy_label
0    0.282288
1    0.430812
2    0.286900
Name: proportion, dtype: float64

side_effects_label distribution
side_effects_label
0    0.552583
1    0.371771
2    0.075646
Name: proportion, dtype: float64
Saved: D:\33333\MedSentiX\data\processed\druglib_clean.csv
Final shape: (1084, 12)


,rating,effectiveness,sideEffects,benefitsReview,sideEffectsReview,commentsReview,review,label,efficacy_label,side_effects_label,ease_label,satisfaction_label
0,4,Highly Effective,Mild Side Effects,slowed the progression of left ventricular dys...,cough hypotension proteinuria impotence renal ...,monitor blood pressure weight and asses for re...,slowed the progression of left ventricular dys...,2,2,1,-100,-100
1,1,Highly Effective,Severe Side Effects,although this type of birth control has more c...,heavy cycle cramps hot flashes fatigue long la...,i hate this birth control i would not suggest ...,although this type of birth control has more c...,0,2,0,-100,-100
3,3,Marginally Effective,Mild Side Effects,the acid reflux went away for a few months aft...,constipation dry mouth and some mild dizziness...,i was given prilosec prescription at a dose of...,the acid reflux went away for a few months aft...,1,1,1,-100,-100
4,2,Marginally Effective,Severe Side Effects,i think that the lyrica was starting to help w...,i felt extremely drugged and dopey could not d...,see above,i think that the lyrica was starting to help w...,0,1,0,-100,-100
5,1,Ineffective,Severe Side Effects,after taking propecia for over a year starting...,low sex drive before i started taking this dru...,one pill once daily,after taking propecia for over a year starting...,0,0,0,-100,-100
